# AWG (Arbitrary Waveform Generator) Driver Test Notebook

This notebook provides a **comprehensive, sequential test** of every function in the AWG driver class hierarchy:

| Layer | Class | Key Methods |
|---|---|---|
| 1 | `Instrument` | `idn()` |
| 2 | `Scpi` | `reset()`, `clear()`, `error()`, `wait()`, `self_test()`, `operation_complete()`, `initialize()` |
| 3 | `Awg` | Output, Waveform, Pulse, Arbitrary, and Trigger methods |

---

**Instructions for the Technician:**
1. Ensure an AWG is connected and powered on.
2. Run each cell **sequentially** from top to bottom.
3. After each cell, verify the **Expected Result** described in the markdown cell that follows.
4. If a cell produces an error or the AWG does not behave as expected, **STOP** and note the failing section.
5. An oscilloscope connected to **Channel 1** output is recommended to visually verify waveform generation.

**Safety Note:** Ensure the AWG output is connected to a proper load (typically 50 Ω or high-Z). Never connect the output to a power supply or other voltage source.

---
## 0. Setup & Connection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from piec.drivers.autodetect import autodetect

In [ ]:
# --- Option A: Auto-detect the AWG ---
awg = autodetect("awg", verbose=True)

# --- Option B: Manually specify the address and driver ---
# from piec.drivers.awg.k_81150a import Keysight81150A
# awg = Keysight81150A("GPIB0::10::INSTR")  # Replace with actual address

**Expected Result:** The cell should print connection information and return without error. If using autodetect, the detected instrument model should be printed.

---
## 1. Instrument-Level Tests (`Instrument` base class)
These tests verify the most fundamental capability: identification.

### 1.1 Identification (`idn`)

In [ ]:
# Query the instrument's identification string
idn_response = awg.idn()
print("IDN Response:")
print(idn_response)

**Expected Result:** A string containing **four comma-separated fields**:  
`Manufacturer, Model, Serial Number, Firmware Version`  

Example: `Agilent Technologies,33220A,MY12345678,2.01-1.00-2.00-1.00`

---
## 2. SCPI-Level Tests (`Scpi` base class)
These tests verify the IEEE 488.2 mandated commands.

### 2.1 Reset (`reset`)

In [ ]:
# Reset the instrument to its factory default state
awg.reset()
print("Reset command sent.")

**Expected Result:** The AWG should visibly return to its **default/factory state**.  
- Waveform reverts to sine  
- Frequency, amplitude reset to defaults  
- Output turns off  

### 2.2 Clear Status (`clear`)

In [ ]:
# Clear the status registers and error queue
awg.clear()
print("Clear command sent.")

**Expected Result:** No visible change on the AWG display. The internal error queue and status registers are cleared.  

### 2.3 Error Query (`error`)

In [ ]:
# Query the error status register
error_response = awg.error()
print("Error Status Register:")
print(error_response)

**Expected Result:** Should return `0` (or `+0`) indicating **no errors** (since we just cleared it).  

### 2.4 Self Test (`self_test`)

In [ ]:
# Run the instrument's internal self-test
# NOTE: This may take several seconds to complete
self_test_result = awg.self_test()
print("Self-test result:")
print(self_test_result)

**Expected Result:** Returns `0` if the self-test **passed**, or a non-zero value if it **failed**.  

**Note:** This command may take 10-30 seconds. The AWG display may flash during the test.  

### 2.5 Wait (`wait`)

In [ ]:
# Wait for all pending operations to complete
awg.wait()
print("Wait command sent.")

**Expected Result:** No visible change. The instrument waits for all pending operations.  

### 2.6 Operation Complete (`operation_complete`)

In [ ]:
# Check if the last operation is complete
opc_result = awg.operation_complete()
print("Operation Complete:")
print(opc_result)

**Expected Result:** Returns `1` indicating all pending operations are complete.  

### 2.7 Initialize (`initialize`)
Convenience method that calls `reset()` followed by `clear()`.

In [ ]:
# Initialize the instrument (reset + clear)
awg.initialize()
print("Initialize command sent (reset + clear).")

**Expected Result:** Same as running reset followed by clear — the AWG returns to factory defaults and the error queue is emptied.  

---
## 3. Output Control (`Awg` class)

### 3.1 Output On (`output`)

In [ ]:
# Turn Channel 1 output ON
awg.output(channel=1, on=True)
print("Channel 1 output ON.")

**Expected Result:** The output indicator for Channel 1 should illuminate (typically a light or on-screen indicator). If a scope is attached, you should see the default waveform.  

### 3.2 Output Off (`output`)

In [ ]:
# Turn Channel 1 output OFF
awg.output(channel=1, on=False)
print("Channel 1 output OFF.")

**Expected Result:** The output indicator for Channel 1 should turn off. If a scope is attached, the signal should disappear (flat line or noise floor).  

---
## 4. Standard Waveform Configuration Tests

### 4.1 Set Waveform (`set_waveform`)

In [ ]:
# Set the waveform to Sine
awg.set_waveform(channel=1, waveform='SIN')
print("Waveform set to SIN.")

**Expected Result:** The AWG display should show the waveform type as **Sine**.  

In [ ]:
# Set the waveform to Square
awg.set_waveform(channel=1, waveform='SQU')
print("Waveform set to SQU.")

**Expected Result:** The AWG display should show the waveform type as **Square**.  

In [ ]:
# Set the waveform to Ramp
awg.set_waveform(channel=1, waveform='RAMP')
print("Waveform set to RAMP.")

**Expected Result:** The AWG display should show the waveform type as **Ramp** (or Triangle).  

### 4.2 Set Frequency (`set_frequency`)

In [ ]:
# Set to a known sine waveform first
awg.set_waveform(channel=1, waveform='SIN')

# Set frequency to 1 kHz
awg.set_frequency(channel=1, frequency=1000)
print("Frequency set to 1 kHz.")

**Expected Result:** The frequency readout on the AWG should show **1.000 kHz** (or 1000 Hz).  

In [ ]:
# Set frequency to 1 MHz
awg.set_frequency(channel=1, frequency=1e6)
print("Frequency set to 1 MHz.")

**Expected Result:** The frequency readout should show **1.000 MHz**.  

### 4.3 Set Amplitude (`set_amplitude`)

In [ ]:
# Set amplitude to 2 Vpp
awg.set_amplitude(channel=1, amplitude=2.0)
print("Amplitude set to 2.0 Vpp.")

**Expected Result:** The amplitude readout should show **2.000 Vpp** (or equivalent).  

In [ ]:
# Set amplitude to 500 mVpp
awg.set_amplitude(channel=1, amplitude=0.5)
print("Amplitude set to 500 mVpp.")

**Expected Result:** The amplitude readout should show **500 mVpp**.  

### 4.4 Set Offset (`set_offset`)

In [ ]:
# Set DC offset to 0.5 V
awg.set_offset(channel=1, offset=0.5)
print("DC offset set to 0.5 V.")

**Expected Result:** The offset readout should show **500 mV** (or 0.5 V). On an oscilloscope, the waveform should shift upward from the zero line.  

In [ ]:
# Reset offset back to 0 V
awg.set_offset(channel=1, offset=0.0)
print("DC offset reset to 0 V.")

**Expected Result:** The offset readout should show **0 V**. The waveform should be centered around zero.  

### 4.5 Set Load Impedance (`set_load_impedance`)

In [ ]:
# Set load impedance to 50 Ohm
awg.set_load_impedance(channel=1, load_impedance=50)
print("Load impedance set to 50 Ohm.")

**Expected Result:** The load impedance setting should show **50 Ω**. This affects how the AWG calculates the output voltage (voltage divider with source impedance).  

**Note:** Some AWGs may not support this command. If unsupported, the driver may raise an error or log a message.  

### 4.6 Set Polarity (`set_polarity`)

In [ ]:
# Set polarity to Inverted
awg.set_polarity(channel=1, polarity='INV')
print("Polarity set to INVERTED.")

**Expected Result:** The waveform output should be **inverted** (180° phase shift). On an oscilloscope, a sine wave should appear flipped.  

In [ ]:
# Set polarity back to Normal
awg.set_polarity(channel=1, polarity='NORM')
print("Polarity set to NORMAL.")

**Expected Result:** The waveform output returns to **normal** (non-inverted).  

### 4.7 Configure Waveform — Combined (`configure_waveform`)

In [ ]:
# Configure a complete waveform in one call
awg.configure_waveform(
    channel=1,
    waveform='SIN',
    frequency=10000,
    amplitude=1.0,
    offset=0.0,
    polarity='NORM'
)
print("Waveform configured: Sine, 10 kHz, 1 Vpp, 0 V offset, Normal polarity.")

**Expected Result:**  
- **Waveform:** Sine  
- **Frequency:** 10 kHz  
- **Amplitude:** 1.0 Vpp  
- **Offset:** 0 V  
- **Polarity:** Normal  

In [ ]:
# Turn output on for visual verification on the oscilloscope
awg.output(channel=1, on=True)
print("Output ON — verify 10 kHz sine wave on oscilloscope.")

**Expected Result:** On the connected oscilloscope, you should see a clean **10 kHz sine wave** with **1 Vpp** amplitude centered at **0 V**.  

---
## 5. Waveform-Specific Parameter Tests

### 5.1 Square Wave — Duty Cycle (`set_square_duty_cycle`)

In [ ]:
# Switch to square wave
awg.set_waveform(channel=1, waveform='SQU')
awg.set_frequency(channel=1, frequency=1000)

# Set duty cycle to 25%
awg.set_square_duty_cycle(channel=1, duty_cycle=25.0)
print("Square wave duty cycle set to 25%.")

**Expected Result:** On an oscilloscope, the square wave should have a **25% duty cycle** — the HIGH portion should be 1/4 of the total period.  

In [ ]:
# Set duty cycle to 75%
awg.set_square_duty_cycle(channel=1, duty_cycle=75.0)
print("Square wave duty cycle set to 75%.")

**Expected Result:** The square wave should now have a **75% duty cycle** — the HIGH portion should be 3/4 of the total period.  

In [ ]:
# Reset to 50%
awg.set_square_duty_cycle(channel=1, duty_cycle=50.0)
print("Square wave duty cycle reset to 50%.")

**Expected Result:** A symmetric square wave (equal HIGH and LOW times).  

### 5.2 Ramp Wave — Symmetry (`set_ramp_symmetry`)

In [ ]:
# Switch to ramp waveform
awg.set_waveform(channel=1, waveform='RAMP')
awg.set_frequency(channel=1, frequency=1000)

# Set symmetry to 50% (triangle wave)
awg.set_ramp_symmetry(channel=1, symmetry=50.0)
print("Ramp symmetry set to 50% (triangle wave).")

**Expected Result:** On an oscilloscope, the waveform should appear as a **symmetric triangle wave** — equal rise and fall times.  

In [ ]:
# Set symmetry to 100% (positive sawtooth)
awg.set_ramp_symmetry(channel=1, symmetry=100.0)
print("Ramp symmetry set to 100% (positive sawtooth).")

**Expected Result:** The waveform should appear as a **positive sawtooth** — slow rise, instant fall.  

In [ ]:
# Set symmetry to 0% (negative sawtooth)
awg.set_ramp_symmetry(channel=1, symmetry=0.0)
print("Ramp symmetry set to 0% (negative sawtooth).")

**Expected Result:** The waveform should appear as a **negative sawtooth** — instant rise, slow fall.  

### 5.3 Pulse Parameters (Individual)

In [ ]:
# Switch to pulse waveform
awg.set_waveform(channel=1, waveform='PULS')
awg.set_frequency(channel=1, frequency=1000)
awg.set_amplitude(channel=1, amplitude=2.0)
print("Switched to Pulse waveform, 1 kHz, 2 Vpp.")

**Expected Result:** The AWG should be set to **Pulse** waveform mode.  

#### 5.3a Set Pulse Width (`set_pulse_width`)

In [ ]:
# Set pulse width to 100 µs
awg.set_pulse_width(channel=1, pulse_width=100e-6)
print("Pulse width set to 100 µs.")

**Expected Result:** The pulse width readout should show **100 µs**. On an oscilloscope, the pulse's HIGH time should be 100 µs out of the 1 ms period (10% duty cycle).  

#### 5.3b Set Pulse Delay (`set_pulse_delay`)

In [ ]:
# Set pulse delay to 50 µs
awg.set_pulse_delay(channel=1, pulse_delay=50e-6)
print("Pulse delay set to 50 µs.")

**Expected Result:** The delay between the start of the period and the leading edge of the pulse should be **50 µs**.  

**Note:** Not all AWGs support pulse delay. If unsupported, an error may be raised.  

#### 5.3c Set Pulse Rise Time (`set_pulse_rise_time`)

In [ ]:
# Set pulse rise time to 10 µs
awg.set_pulse_rise_time(channel=1, rise_time=10e-6)
print("Pulse rise time set to 10 µs.")

**Expected Result:** The leading edge of the pulse should have a **10 µs** transition time. On an oscilloscope, the rising edge should appear sloped rather than instantaneous.  

#### 5.3d Set Pulse Fall Time (`set_pulse_fall_time`)

In [ ]:
# Set pulse fall time to 10 µs
awg.set_pulse_fall_time(channel=1, fall_time=10e-6)
print("Pulse fall time set to 10 µs.")

**Expected Result:** The trailing edge of the pulse should have a **10 µs** transition time. The falling edge should appear sloped.  

#### 5.3e Set Pulse Duty Cycle (`set_pulse_duty_cycle`)

In [ ]:
# Set pulse duty cycle to 30%
awg.set_pulse_duty_cycle(channel=1, duty_cycle=30.0)
print("Pulse duty cycle set to 30%.")

**Expected Result:** The pulse HIGH time should be **30%** of the total period.  

### 5.4 Configure Pulse — Combined (`configure_pulse`)

In [ ]:
# Configure all pulse parameters in one call
awg.configure_pulse(
    channel=1,
    pulse_width=200e-6,
    rise_time=5e-6,
    fall_time=5e-6,
    duty_cycle=20.0
)
print("Pulse configured: 200 µs width, 5 µs rise/fall, 20% duty cycle.")

**Expected Result:**  
- **Waveform:** Pulse (set automatically by `configure_pulse`)  
- **Width:** 200 µs  
- **Rise Time:** 5 µs  
- **Fall Time:** 5 µs  
- **Duty Cycle:** 20%  

---
## 6. Arbitrary Waveform Tests

### 6.1 Create Arbitrary Waveform (`create_arb_waveform`)

In [ ]:
# Create a simple staircase arbitrary waveform
# Data is normalized to the AWG's arb data range — check your driver for specifics
num_points = 1000
steps = 5
staircase = np.repeat(np.linspace(-1, 1, steps), num_points // steps)

plt.figure(figsize=(10, 3))
plt.plot(staircase)
plt.title("Staircase Waveform (to be uploaded)")
plt.ylabel("Normalized Amplitude")
plt.xlabel("Sample Index")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Waveform has {len(staircase)} points.")

**Expected Result:** A matplotlib plot should appear showing a **5-step staircase** waveform. This is the waveform we will upload to the AWG.

In [ ]:
# Upload the arbitrary waveform to the AWG
awg.create_arb_waveform(channel=1, name='STAIRCASE', data=staircase)
print("Arbitrary waveform 'STAIRCASE' created.")

**Expected Result:** The waveform data should be transferred to the AWG's memory without error. Some AWGs may show the waveform name in their memory/catalog.  

### 6.2 Set Arbitrary Waveform (`set_arb_waveform`)

In [ ]:
# Select the uploaded arbitrary waveform for output
awg.set_arb_waveform(channel=1, name='STAIRCASE')
awg.set_frequency(channel=1, frequency=100)
awg.output(channel=1, on=True)
print("Arbitrary waveform 'STAIRCASE' selected and output ON at 100 Hz.")

**Expected Result:** On an oscilloscope connected to the AWG output, you should see a **staircase pattern**. The waveform should repeat at **100 Hz**.  

---
## 7. Trigger Configuration Tests

In [ ]:
# Switch back to a standard waveform for trigger testing
awg.configure_waveform(
    channel=1,
    waveform='SIN',
    frequency=1000,
    amplitude=1.0,
    offset=0.0
)
print("Reset to Sine 1 kHz, 1 Vpp for trigger tests.")

### 7.1 Set Trigger Source (`set_trigger_source`)

In [ ]:
# Set trigger source to Immediate (continuous/free-run)
awg.set_trigger_source(channel=1, trigger_source='IMM')
print("Trigger source set to IMM (Immediate/Continuous).")

**Expected Result:** The AWG should output continuously (free-running) with no external trigger required.  

In [ ]:
# Set trigger source to External
awg.set_trigger_source(channel=1, trigger_source='EXT')
print("Trigger source set to EXT (External).")

**Expected Result:** The AWG should now wait for an external trigger signal on the trigger input connector. The output may stop generating if no external trigger is present.  

In [ ]:
# Set trigger source back to Immediate for remaining tests
awg.set_trigger_source(channel=1, trigger_source='IMM')
print("Trigger source reset to IMM.")

### 7.2 Set Trigger Slope (`set_trigger_slope`)

In [ ]:
# Set trigger slope to Positive (Rising edge)
awg.set_trigger_slope(channel=1, trigger_slope='POS')
print("Trigger slope set to POS (rising edge).")

**Expected Result:** The trigger slope should be set to **Positive** (rising edge). This determines which edge of the external trigger signal starts a burst.  

### 7.3 Set Trigger Mode (`set_trigger_mode`)

In [ ]:
# Set trigger mode to Edge
awg.set_trigger_mode(channel=1, trigger_mode='EDGE')
print("Trigger mode set to EDGE.")

**Expected Result:** The trigger mode should be set to **Edge**.  

### 7.4 Configure Trigger — Combined (`configure_trigger`)

In [ ]:
# Configure all trigger settings in one call
awg.configure_trigger(
    channel=1,
    trigger_source='IMM',
    trigger_slope='POS',
    trigger_mode='EDGE'
)
print("Trigger configured: IMM source, POS slope, EDGE mode.")

**Expected Result:**  
- **Source:** Immediate (continuous)  
- **Slope:** Positive  
- **Mode:** Edge  

### 7.5 Output Trigger / Software Trigger (`output_trigger`)

In [ ]:
# First set trigger to Manual so the AWG waits for a software trigger
awg.set_trigger_source(channel=1, trigger_source='MAN')
print("Trigger source set to MAN (manual/software).")

# Now send a software trigger
awg.output_trigger()
print("Software trigger sent.")

**Expected Result:** The AWG should fire a single trigger event (or burst, depending on burst configuration). On an oscilloscope, you should see a single waveform cycle or burst appear.  

In [ ]:
# Reset trigger back to Immediate for continuous output
awg.set_trigger_source(channel=1, trigger_source='IMM')
print("Trigger source reset to IMM.")

---
## 8. Optional Feature Tests
These features are `@optional` — they will gracefully skip if the connected AWG does not support them.

### 8.1 Burst Mode (`configure_burst`)

In [ ]:
# Configure burst mode: 5 cycles per trigger
awg.set_waveform(channel=1, waveform='SIN')
awg.set_frequency(channel=1, frequency=1e3)
awg.set_amplitude(channel=1, amplitude=1.0)

awg.configure_burst(channel=1, burst_mode='TRIG', burst_count=5)
print("Burst mode: TRIG, 5 cycles per trigger.")

**Expected Result:** AWG shows burst mode enabled with **5 cycles** per trigger event. On a scope (single-shot), you should see exactly 5 sine cycles when triggered.

✅ **PASS** if burst is configured, or gracefully skips.

### 8.2 Phase Control (`set_phase`)

In [ ]:
# Set phase offset to 90 degrees
awg.set_phase(channel=1, phase=90)
print("Phase set to 90°.")

**Expected Result:** The waveform's starting phase shifts by **90°**. On a dual-channel scope, CH1 should visibly lead/lag vs a reference.

✅ **PASS** if phase changes, or gracefully skips.

In [ ]:
# Reset phase to 0
awg.set_phase(channel=1, phase=0)
print("Phase reset to 0°.")

### 8.3 Frequency Sweep (`configure_sweep`)

In [ ]:
# Configure a linear sweep from 100 Hz to 10 kHz over 1 second
awg.configure_sweep(
    channel=1,
    sweep_mode='LIN',
    start_freq=100,
    stop_freq=10e3,
    sweep_time=1.0
)
print("Sweep: LIN, 100 Hz → 10 kHz, 1 s.")

**Expected Result:** AWG display shows sweep mode enabled. On a scope, the frequency should visibly increase from 100 Hz to 10 kHz over 1 second.

✅ **PASS** if sweep is configured, or gracefully skips.

### 8.4 Modulation (`configure_modulation`)

In [ ]:
# Configure AM modulation: 50% depth, 100 Hz internal source
awg.configure_modulation(
    channel=1,
    mod_type='AM',
    depth=50,
    frequency=100,
    source='INT'
)
print("Modulation: AM, 50% depth, 100 Hz internal.")

**Expected Result:** AWG shows AM modulation enabled. On a scope, the sine wave amplitude should visibly oscillate at 100 Hz.

✅ **PASS** if modulation is configured, or gracefully skips.

---
## 9. Cleanup & Final Reset

In [ ]:
# Turn off output and reset
awg.output(channel=1, on=False)
awg.reset()
print("Output OFF. AWG reset to factory defaults.")

**Expected Result:** The AWG output turns off and the instrument returns to its default/factory state.  